In [3]:
# Cell 1 — setup
import sys, os
sys.path.append(os.path.abspath('..'))

In [4]:
# Cell 2 — funnel analysis
from src.db import get_funnel_conversion, get_funnel_by_segment
import plotly.express as px

funnel_df = get_funnel_conversion()
print(funnel_df[['stage','sessions','step_conversion_pct',
                  'dropoff_pct','estimated_lost_revenue']])

         stage  sessions  step_conversion_pct  dropoff_pct  \
0         view    200000                  NaN          NaN   
1  add_to_cart     70080                 35.0         65.0   
2     checkout     13475                 19.2         80.8   
3     purchase      1826                 13.6         86.4   

   estimated_lost_revenue  
0            0.000000e+00  
1            4.332957e+09  
2            1.887831e+09  
3            3.885053e+08  


In [5]:
# Cell 3 — funnel chart
fig = px.funnel(funnel_df, x='sessions', y='stage',
                title='Customer purchase funnel — overall conversion')
fig.write_html("funnel_chart.html")
print("Saved! Open funnel_chart.html")

Saved! Open funnel_chart.html


In [6]:
# Cell 4 — which stage loses the most revenue?
worst = funnel_df.loc[funnel_df['estimated_lost_revenue'].idxmax()]
print(f"\nBiggest drop-off stage : {worst['stage']}")
print(f"Sessions lost          : {worst['sessions_dropped']:,.0f}")
print(f"Estimated lost revenue : ₹{worst['estimated_lost_revenue']:,.2f}")


Biggest drop-off stage : add_to_cart
Sessions lost          : 129,920
Estimated lost revenue : ₹4,332,956,839.67


In [7]:
# Cell 5 — funnel by acquisition channel
seg_df = get_funnel_by_segment()
purchase = seg_df[seg_df['stage'] == 'purchase'].sort_values(
    'overall_conv_pct', ascending=False)
print("\nBest converting channels:")
print(purchase[['channel','city_tier','overall_conv_pct']].head(10))


Best converting channels:
           channel city_tier  overall_conv_pct
35  Email Campaign    Tier-3               1.2
31  Email Campaign    Tier-2               1.1
59        Paid Ads    Tier-3               1.1
23          Direct    Tier-3               1.0
11             App    Tier-3               1.0
63        Referral    Tier-1               1.0
67        Referral    Tier-2               1.0
47  Organic Search    Tier-3               1.0
7              App    Tier-2               0.9
75    Social Media    Tier-1               0.9


In [8]:
# Cell 5 — funnel by acquisition channel
seg_df = get_funnel_by_segment()
purchase = seg_df[seg_df['stage'] == 'purchase'].sort_values(
    'overall_conv_pct', ascending=False)
print("\nBest converting channels:")
print(purchase[['channel','city_tier','overall_conv_pct']].head(10))


Best converting channels:
           channel city_tier  overall_conv_pct
35  Email Campaign    Tier-3               1.2
31  Email Campaign    Tier-2               1.1
59        Paid Ads    Tier-3               1.1
23          Direct    Tier-3               1.0
11             App    Tier-3               1.0
63        Referral    Tier-1               1.0
67        Referral    Tier-2               1.0
47  Organic Search    Tier-3               1.0
7              App    Tier-2               0.9
75    Social Media    Tier-1               0.9


In [9]:
# Cell 6 — channel conversion chart
fig2 = px.bar(purchase, x='channel', y='overall_conv_pct',
              color='city_tier',
              title='Purchase conversion % by channel and city tier',
              labels={'overall_conv_pct':'Conversion %',
                      'channel':'Acquisition channel'},
              barmode='group')
fig2.write_html("funnel_by_channel.html")
print("Saved! Open funnel_by_channel.html")

Saved! Open funnel_by_channel.html


In [10]:
# Cell 7 — product pareto
from src.db import get_product_pareto, get_category_revenue

pareto_df = get_product_pareto()
top_80 = pareto_df[pareto_df['is_top_80pct_driver'] == 1]
print(f"\nTotal products        : {len(pareto_df)}")
print(f"Products driving 80%  : {len(top_80)}")
print(f"That is top           : {len(top_80)/len(pareto_df)*100:.1f}% of catalogue")


Total products        : 500
Products driving 80%  : 149
That is top           : 29.8% of catalogue


In [11]:
# Cell 8 — pareto chart
fig3 = px.bar(pareto_df.head(30), x='product_name', y='revenue',
              color='is_top_80pct_driver',
              color_discrete_map={1:'#1D9E75', 0:'#B4B2A9'},
              title='Top 30 products by revenue — green = top 80% drivers')
fig3.add_scatter(x=pareto_df.head(30)['product_name'],
                 y=pareto_df.head(30)['cumulative_revenue_pct'],
                 name='Cumulative %', yaxis='y2',
                 line=dict(color='#534AB7', width=2))
fig3.update_layout(
    yaxis2=dict(overlaying='y', side='right',
                title='Cumulative revenue %', range=[0,100]))
fig3.write_html("pareto_chart.html")
print("Saved! Open pareto_chart.html")

Saved! Open pareto_chart.html


In [12]:
# Cell 9 — category breakdown
cat_df = get_category_revenue()
fig4 = px.bar(cat_df, x='category', y='revenue',
              color='revenue_share_pct',
              title='Revenue by product category',
              labels={'revenue':'Revenue (₹)','category':'Category'},
              text='revenue_share_pct')
fig4.write_html("category_revenue.html")
print("Saved! Open category_revenue.html")

Saved! Open category_revenue.html


In [13]:
# Cell 10 — write your business insight
print("=" * 55)
print("NOTEBOOK 4 KEY FINDINGS")
print("=" * 55)

worst_stage = funnel_df.loc[funnel_df['dropoff_pct'].fillna(0).idxmax()]
best_channel = purchase.iloc[0]

print(f"""
Funnel:
  - Biggest drop-off : {worst_stage['stage']} 
    ({worst_stage['dropoff_pct']}% of users leave here)
  - Lost revenue     : ₹{worst_stage['estimated_lost_revenue']:,.0f}
  - Best channel     : {best_channel['channel']} 
    ({best_channel['overall_conv_pct']}% conversion)

Pareto:
  - {len(top_80)} products ({len(top_80)/len(pareto_df)*100:.0f}% of catalogue) 
    drive 80% of revenue
  - Top category     : {cat_df.iloc[0]['category']} 
    ({cat_df.iloc[0]['revenue_share_pct']}% of total revenue)
""")

NOTEBOOK 4 KEY FINDINGS

Funnel:
  - Biggest drop-off : purchase 
    (86.4% of users leave here)
  - Lost revenue     : ₹388,505,343
  - Best channel     : Email Campaign 
    (1.2% conversion)

Pareto:
  - 149 products (30% of catalogue) 
    drive 80% of revenue
  - Top category     : Electronics 
    (72.36% of total revenue)

